# Basic Cleaning
Cleans the raw RTA dataset.

In [28]:
# imports
import pandas as pd
import numpy as np

In [29]:
# load raw data
df = pd.read_csv("../Dataset.csv")
df.shape

(6178, 74)

In [30]:
# drop identity / contact / address columns (PII, not useful for modeling)
pii_cols = ["Person Id", "Full Name", "Contact Number", "Address Street", "Ethnicity", "Hospital Reference Number"]
df = df.drop(columns=[c for c in pii_cols if c in df.columns])

In [31]:
# drop columns that only ever have one value (they carry no information)
constant_cols = [c for c in df.columns if not c.endswith("_missing_reason") and df[c].nunique(dropna=True) <= 1]
df = df.drop(columns=constant_cols)
constant_cols

['Facilities for Daily Activity', 'First Aid Given At Seen']

In [32]:
# strip stray spaces and standardize case on every text column, so the same value
# typed differently (extra spaces, different capitalization) isn't treated as two categories
text_cols = df.select_dtypes(include='object').columns

for c in text_cols:
    df[c] = df[c].astype(str).str.strip().str.lower()
    df.loc[df[c].isin(['nan', 'none']), c] = np.nan

C:\Users\abaiy\AppData\Local\Temp\ipykernel_29076\321737981.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include='object').columns


In [33]:
# known placeholder phrases used instead of real NaN in this dataset
# (keys are lowercase - the normalization step above lowercases the data before this runs)
placeholder_map = {
    "victim not willing to share/ unable to respond/  early discharge": "patient_unable_unwilling",
    "victim doesn't have knowledge on distance/ not willing to share/ unable to respond/  early discharge" : "patient_has_no_knowledge",
    "victim unable to recall the time or early discharge": "patient_unable_unwilling",
    "victim not aware about the time/ not willing to share/ unable to respond/  early discharge": "patient_unable_unwilling",
    "early discharge/ unable to look the bht": "early_discharge",
    "early discharge/ unable to access the bht": "early_discharge",
    "data missing": "data_entry_gap",
    "missing data": "data_entry_gap",
    "not necessary": "not_applicable",
    "not necessary for student and unemployed" : "not_applicable",
    "na": "not_applicable",
    "unknown": "data_entry_gap"
}

In [34]:
# for every column, replace placeholder text with NaN and store the reason in a new column
for col in df.columns:
    reason_col = col + "_missing_reason"
    df[reason_col] = "not_missing"
    for phrase, reason in placeholder_map.items():
        mask = df[col] == phrase
        df.loc[mask, reason_col] = reason
        df.loc[mask, col] = np.nan
    # plain blank cells are also missing, just a data entry gap
    blank_mask = df[col].isna() & (df[reason_col] == "not_missing")
    df.loc[blank_mask, reason_col] = "data_entry_gap"

In [35]:
# remove trailing "y" from age range values (e.g. "1-9y" -> "1-9")
df["Age"] = df["Age"].astype(str).str.replace("y", "", regex=False)

In [36]:
# save cleaned dataset for use in the model notebooks
df.to_csv("Preprocessed Dataset.csv", index=False)
df.shape

(6178, 132)